# Medical Span Pseudo-Perplexity

Computes pseudo-perplexity **restricted to medical NER spans** (DISEASE, PROCEDURE, SYMPTOM, MEDICATION)
identified on the ParaClite dataset, using `SpanPPLBackend` and `SpanMiniconsStridedBackend`.

Each span is scored **in context**: the surrounding document is fully visible to the model;
only the span's own tokens are masked and scored one at a time.

In [1]:
import gc
import json
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch

import sys
sys.path.append("../src")
from pppl.backends_medical_span import SpanPPLBackend, SpanMiniconsStridedBackend
from pppl.span_utils import load_predictions, load_paraclite_docs, build_span_dataset

## Configuration

Set `LANGUAGE` to the target language column in paraclite.csv and pick the
matching NER prediction file and language model(s).

In [2]:
LANGUAGE = 'ro' 

PARACLITE_CSV   = "../data/paraclite.csv"
PREDICTIONS_DIR = "../../paraclite_inference_results"
OUTPUT_DIR      = "../output_medical"

# NER prediction file — use multilabel for all entity types at once.
# Swap to a multiclass file (e.g. *_dis_predictions.tsv) to focus on one type.
PREDICTIONS_FILE = f"{PREDICTIONS_DIR}/{LANGUAGE}_multilabel_xlm_roberta_large_all_entities_predictions.tsv"

# Language models for span perplexity scoring.
MODEL_LIST = [
    '/mnt/DATA/DT4H/NER_task/CardioLM_paper_models/base_models/CardioBERTa_ro',
    '/mnt/DATA/DT4H/NER_task/CardioLM_paper_models/base_models/xlm-roberta-base',
    # '/mnt/DATA/DT4H/NER_task/CardioLM_paper_models/base_models/CardioBERTa_cz',
    # '/mnt/DATA/DT4H/NER_task/CardioLM_paper_models/base_models/robeczech-base',
]

# Short display name for each model path (used as column suffixes).
MODEL_SHORT_NAMES = {
    '/mnt/DATA/DT4H/NER_task/CardioLM_paper_models/base_models/CardioBERTa_ro'          : 'CardioBERTa_ro',
    '/mnt/DATA/DT4H/NER_task/CardioLM_paper_models/base_models/CardioBERTa_cz'          : 'CardioBERTa_cz',
    '/mnt/DATA/DT4H/NER_task/CardioLM_paper_models/base_models/robeczech-base'          : 'robeczech-base',
    '/mnt/DATA/DT4H/NER_task/CardioLM_paper_models/base_models/xlm-roberta-base'        : 'xlm-roberta-base',
}

## Load Data

In [3]:
docs    = load_paraclite_docs(PARACLITE_CSV, language=LANGUAGE)
spans   = load_predictions(PREDICTIONS_FILE)
span_df = build_span_dataset(spans, docs)

# Source language derived from the document filename prefix (e.g. 'ro_patient_4' → 'ro').
span_df['src_lang'] = span_df['filename'].str.split('_').str[0]

print(f"Spans: {len(span_df):,} across {span_df['filename'].nunique()} documents")
print(f"Target language: {LANGUAGE}")
print()
print(span_df['label'].value_counts().to_string())

Spans: 6,781 across 94 documents
Target language: ro

label
DISEASE       2217
SYMPTOM       2050
PROCEDURE     1643
MEDICATION     871


In [4]:
# Quick sanity-check: verify that text[start_span:end_span] matches the predicted span text.
sample = span_df.sample(5, random_state=42)
for _, row in sample.iterrows():
    extracted = row['doc_text'][row['start_span']:row['end_span']]
    match = extracted.strip() == str(row['text']).strip()
    print(f"{'OK' if match else 'MISMATCH'}  [{row['label']}]  {repr(str(row['text'])[:60])}")

OK  [SYMPTOM]  'VCI subțire'
OK  [PROCEDURE]  'EKG'
OK  [MEDICATION]  'Nexium'
OK  [PROCEDURE]  'cardioversie'
OK  [DISEASE]  'blocului AV'


## Scoring Utilities

In [5]:
def _free(scorer):
    """Best-effort GPU release for any backend."""
    for attr_chain in (("model",), ("_scorer", "model")):
        obj = scorer
        for a in attr_chain:
            obj = getattr(obj, a, None)
            if obj is None:
                break
        if obj is not None and hasattr(obj, "to"):
            try:
                obj.to("cpu")
            except Exception:
                pass
    del scorer
    gc.collect()
    torch.cuda.empty_cache()

## Span Perplexity — PPLBackend

Masks each span token once, with the full document as context.
Results are added as columns `ppl_<model_short_name>`.

In [7]:
for model_name in MODEL_LIST:
    short = MODEL_SHORT_NAMES.get(model_name, model_name)
    col   = f'ppl_{short}'
    print(f"[PPLBackend] {short}")

    scorer = SpanPPLBackend(
        model_name,
        device='cuda',
        windows_size=508,
        stride=256,
        batch_size=8,
    )
    span_df[col] = scorer.score_spans(span_df, show_progress=True)
    _free(scorer)

    print(f"  mean={span_df[col].mean():.3f}  median={span_df[col].median():.3f}  "
          f"NaN={span_df[col].isna().sum()}")
    print()

[PPLBackend] CardioBERTa_ro


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

SpanPPLBackend: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 6781/6781 [34:26<00:00,  3.28it/s]


  mean=1751.919  median=1.463  NaN=2

[PPLBackend] xlm-roberta-base


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] XLMRobertaForMaskedLM LOAD REPORT from: /mnt/DATA/DT4H/NER_task/CardioLM_paper_models/base_models/xlm-roberta-base
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
SpanPPLBackend: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 6781/6781 [33:57<00:00,  3.33it/s]


  mean=238715.102  median=4.292  NaN=2



## Span Perplexity — MiniconsStridedBackend

Results are added as columns `minicons_<model_short_name>`.

In [8]:
for model_name in MODEL_LIST:
    short = MODEL_SHORT_NAMES.get(model_name, model_name)
    col   = f'minicons_{short}'
    print(f"[MiniconsStridedBackend] {short}")

    scorer = SpanMiniconsStridedBackend(
        model_name,
        device='cuda',
        windows_size=508,
        stride=256,
        model_batch_size=8,
        pll_method='within_word_l2r',
    )
    span_df[col] = scorer.score_spans(span_df, show_progress=True)
    _free(scorer)

    print(f"  mean={span_df[col].mean():.3f}  median={span_df[col].median():.3f}  "
          f"NaN={span_df[col].isna().sum()}")
    print()

[MiniconsStridedBackend] CardioBERTa_ro


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

SpanMiniconsStridedBackend: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 6781/6781 [29:11<00:00,  3.87it/s]


  mean=1616.019  median=8.119  NaN=2

[MiniconsStridedBackend] xlm-roberta-base


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

SpanMiniconsStridedBackend: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 6781/6781 [28:43<00:00,  3.93it/s]


  mean=54716.896  median=31.429  NaN=2



## Save Results

In [9]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Drop the (potentially large) full doc text before saving to keep file size manageable.
save_cols = [c for c in span_df.columns if c != 'doc_text']
out_path  = f"{OUTPUT_DIR}/span_ppl_{LANGUAGE}.csv"
span_df[save_cols].to_csv(out_path, index=False)
print(f"Saved {len(span_df):,} rows → {out_path}")

Saved 6,781 rows → ../output_medical/span_ppl_ro.csv


## Analysis

In [23]:
def geo_mean(s):
    return np.exp(np.log(s.dropna()).mean())

In [24]:
ppl_cols = [c for c in span_df.columns if c.startswith('ppl_') or c.startswith('minicons_')]

print("=== Overall aggregate (all languages, all entity types) ===")
span_df[ppl_cols].agg(['mean', 'median', geo_mean]).round(3)

=== Overall aggregate (all languages, all entity types) ===


,ppl_CardioBERTa_ro,ppl_xlm-roberta-base,minicons_CardioBERTa_ro,minicons_xlm-roberta-base
mean,1751.919,238715.102,1616.019,54716.896
median,1.463,4.292,8.119,31.429
geo_mean,3.514,9.787,13.057,42.171


In [25]:
print("=== Aggregate by entity type ===")
span_df.groupby('label')[ppl_cols].agg(['mean', 'median', geo_mean]).round(3)

=== Aggregate by entity type ===


ppl_CardioBERTa_ro                 ppl_xlm-roberta-base          \
                         mean median geo_mean                 mean  median   
label                                                                        
DISEASE              1051.579  1.259    2.894            97829.561   2.908   
MEDICATION           2617.438  2.309   11.049            83673.225  41.382   
PROCEDURE            3992.627  1.381    2.768            32043.570   4.384   
SYMPTOM               346.570  1.693    3.231           622439.963   4.436   

                    minicons_CardioBERTa_ro                   \
           geo_mean                    mean  median geo_mean   
label                                                          
DISEASE       6.946                1231.568   5.319    9.634   
MEDICATION   47.502                3402.709  33.960   53.199   
PROCEDURE     7.930                1950.779   7.232   10.645   
SYMPTOM       8.592                1006.109   8.778   11.780   

           minicons_xlm-roberta-base                    
                                mean   median geo_mean  
label                                                   
DISEASE                     7892.044   23.634   31.200  
MEDICATION                 32019.370  220.062  168.671  
PROCEDURE                   1759.391   32.290   39.119  
SYMPTOM                   157421.299   25.972   34.473

In [26]:
print("=== Aggregate by source language ===")
span_df.groupby('src_lang')[ppl_cols].agg(['mean', 'median', geo_mean]).round(3)

=== Aggregate by source language ===


ppl_CardioBERTa_ro                 ppl_xlm-roberta-base          \
                       mean median geo_mean                 mean  median   
src_lang                                                                   
cs                  990.232  1.542    3.861            86607.020   4.295   
en                  597.165  1.050    1.506            17858.640   2.498   
es                   19.934  1.709    2.637              844.417   3.309   
it                 3059.929  1.636    5.102          1000592.029   6.342   
nl                 5530.003  1.290    2.969            64687.878   3.444   
ro                  965.746  5.231   10.075            93234.082  14.459   
sv                  781.317  1.158    2.787           405390.778   3.357   

                  minicons_CardioBERTa_ro                   \
         geo_mean                    mean  median geo_mean   
src_lang                                                     
cs         11.064                1811.481   9.135   14.519   
en          4.889                 504.089   3.354    4.598   
es          5.668                 255.409   5.067    6.970   
it         15.458                2847.309  11.802   21.129   
nl          8.275                3278.209   6.120   10.260   
ro         24.239                1576.126  36.524   50.251   
sv          8.800                 919.097   7.633   12.062   

         minicons_xlm-roberta-base                    
                              mean   median geo_mean  
src_lang                                              
cs                        2904.173   35.757   49.734  
en                        3628.277   16.159   19.716  
es                         359.062   14.389   16.443  
it                       20617.700   62.614   81.290  
nl                        5563.145   24.995   33.141  
ro                       16754.762  107.734  117.328  
sv                      295938.139   30.219   44.007

In [27]:
print("=== Aggregate by entity type × source language ===")
span_df.groupby(['label', 'src_lang'])[ppl_cols].agg(['mean', 'median', geo_mean]).round(3)

=== Aggregate by entity type × source language ===


ppl_CardioBERTa_ro                   ppl_xlm-roberta-base  \
                                  mean   median geo_mean                 mean   
label      src_lang                                                             
DISEASE    cs                   54.747    1.241    2.446           245787.654   
           en                 1789.307    1.022    1.436              254.193   
           es                   17.677    1.616    2.379             1192.270   
           it                 1059.903    1.257    3.394             6092.043   
           nl                  933.237    1.298    2.771             5856.318   
           ro                 1770.781    4.722    8.071           281876.020   
           sv                 1673.601    1.045    2.580           159998.468   
MEDICATION cs                 7393.532   94.125   93.315            18975.255   
           en                    4.389    1.018    1.507             7668.059   
           es                   48.181    1.327    2.726              531.170   
           it                 4093.057   85.066   73.152            24940.374   
           nl                 4316.229    1.146    2.766           451711.247   
           ro                 2271.913  116.385   86.550            11480.172   
           sv                  953.346    1.123    5.481             2796.780   
PROCEDURE  cs                  493.027    1.179    2.341            36930.759   
           en                    3.625    1.055    1.534            13428.845   
           es                   17.953    1.785    2.615              890.216   
           it                 6585.213    1.362    3.360             2784.103   
           nl                16522.795    1.203    3.053             2320.591   
           ro                   90.880    2.798    4.976              459.827   
           sv                  147.182    1.440    2.391           137405.239   
SYMPTOM    cs                  238.437    1.638    3.044              852.092   
           en                    4.221    1.131    1.565            44998.955   
           es                   15.697    1.895    2.911              556.064   
           it                 1791.651    1.754    3.690          3670910.335   
           nl                   50.409    1.724    3.268              236.738   
           ro                  249.879    5.062    8.817             2090.203   
           sv                  302.614    1.381    2.359          1218168.634   

                                      minicons_CardioBERTa_ro           \
                      median geo_mean                    mean   median   
label      src_lang                                                      
DISEASE    cs          2.903    7.282                 383.765    6.154   
           en          1.818    3.290                1361.864    2.315   
           es          3.159    5.020                 639.211    4.329   
           it          3.729    9.812                1372.497    7.554   
           nl          2.788    6.640                1343.097    4.700   
           ro         10.144   15.239                2111.861   29.171   
           sv          1.587    6.409                1330.547    3.381   
MEDICATION cs        262.775  209.971               13463.584  431.994   
           en         10.080   21.108                 141.842   12.828   
           es          2.089    6.913                  92.152    3.473   
           it        226.876  219.062                3875.093  404.776   
           nl          4.232   15.481                3170.636   10.090   
           ro        280.056  229.196                3803.070  272.667   
           sv         10.923   27.963                1344.273   23.448   
PROCEDURE  cs          3.907    8.038                 722.656    6.156   
           en          3.289    5.285                 122.637    3.495   
           es          3.143    5.457                  33.740    3.862   
           it          4.553